In [4]:
from pathlib import Path

import numpy as np
import torch

from model.RadViT import RadViT

def load_files(folder, ext):
    files = sorted(folder.glob(f"*{ext}"))
    times = np.array([float(f.stem) for f in files])
    return files, times

def get_complex_content(file):
    data = open(file, "rb").read()
    arr = np.frombuffer(data, dtype=np.uint16)

    content = np.empty((3, 256, 256), dtype="complex")
    size = 2 * 256 * 256
    for i in range(3):
        sub = arr[i*size:(i+1)*size]
        content[i] = (sub[0::2] + 1j * sub[1::2]).reshape((256, 256))
    return content


raw_files, raw_times = load_files(Path("/Benson_DATA3/Public/MUSE/data_route_2_camionette/raw"), ".raw")
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
parameters = {
    "D": 256,
    "p": 16,
    "H": 256,
    "W": 256,
    "neuron": 4,
    "mha": 8,
    "layer": 6,
    "dropout": 0.2,
    "n_encoders": 1
}
net = RadViT(parameters['D'], parameters['p'], parameters['H'], parameters['W'], parameters['neuron'], parameters['mha'], parameters['layer'], parameters['dropout'], parameters['n_encoders'], kmd2=True, data_mode="ADC")
net.to(device)

checkpoint = torch.load("/home/skouff/master_thesis/experiments/RadViT_ADC_Class/ADC_MVIT_AP_0.8786_AR_0.8056_F1_0.8406_best.pth", weights_only=False, map_location='cpu')
model_state_dict = {k.replace('module.', ''): v for k, v in checkpoint['net_state_dict'].items()}

net.load_state_dict(model_state_dict, strict=False)
net.eval()

for i in range(len(raw_files)):
    content = get_complex_content(raw_files[i])
    content = torch.from_numpy(content).unsqueeze(0).to(device)
    with torch.no_grad():
        output = net(content)
    

RuntimeError: Error(s) in loading state_dict for RadViT:
	size mismatch for DFT.range_net.range_nn.weight.imag: copying a param with shape torch.Size([512, 512]) from checkpoint, the shape in current model is torch.Size([256, 256]).
	size mismatch for DFT.range_net.range_nn.weight.real: copying a param with shape torch.Size([512, 512]) from checkpoint, the shape in current model is torch.Size([256, 256]).